# Proyecto · Clase 5 — La red de verdad: una capa oculta y retropropagación

Las 10 neuronas de la Clase 4 se quedan en ~92 %: cada una solo puede separar con **una línea recta**.

Solución: poner **en medio** una capa de neuronas que aprendan a detectar *rasgos* (trazos, curvas, bucles) y que
las 10 neuronas de salida decidan **con esos rasgos** en vez de con píxeles sueltos.

```
imagen (784)  →  capa oculta (128, ReLU)  →  salida (10, softmax)
```

El reto: ¿cómo se corrigen los pesos de una capa que **no tiene respuesta correcta**? → **Retropropagación**.

In [ ]:
import os, urllib.request
for archivo in ["digitos.py", "dibujar.py"]:
    if not os.path.exists(archivo):
        urllib.request.urlretrieve("https://raw.githubusercontent.com/datahuba/curso-redes-neuronales/main/proyecto/" + archivo, archivo)

import numpy as np
import matplotlib.pyplot as plt
from digitos import cargar_mnist, mostrar, ok, guardar_pesos, leer_dibujo

X_train, y_train, X_test, y_test = cargar_mnist()
X, Xt = X_train.reshape(-1, 784) / 255, X_test.reshape(-1, 784) / 255
Y = np.eye(10)[y_train]


def softmax(Z):
    E = np.exp(Z - Z.max(axis=1, keepdims=True))
    return E / E.sum(axis=1, keepdims=True)

## 1. ReLU: la activación de la capa oculta
$$\text{ReLU}(z) = \max(0, z)$$
Deja pasar lo positivo y apaga lo negativo. Es simple, rápida, y **sin ella dos capas serían lo mismo que una**
(multiplicar matrices seguidas sigue siendo una recta).

✏️ **Ejercicio 1.**

In [ ]:
def relu(Z):
    return ...

In [ ]:
ok(relu(np.array([-2.0, 0.0, 3.0])).tolist() == [0, 0, 3], "ReLU apaga los negativos")

## 2. Los pesos de dos capas
Pesos iniciales **al azar y pequeños** (si todos empiezan en 0, todas las neuronas ocultas aprenderían lo mismo).

In [ ]:
def crear_red(ocultas=128, semilla=0):
    rng = np.random.default_rng(semilla)
    return {"W1": rng.normal(0, np.sqrt(2 / 784), (784, ocultas)), "b1": np.zeros(ocultas),
            "W2": rng.normal(0, np.sqrt(2 / ocultas), (ocultas, 10)), "b2": np.zeros(10)}

red = crear_red()
for k, v in red.items():
    print(k, v.shape)

## 3. Hacia adelante (*forward*)
✏️ **Ejercicio 2.** Calcula paso a paso. Guardamos los resultados intermedios porque la retropropagación los usa.
```
Z1 = X·W1 + b1     A1 = relu(Z1)     P = softmax(A1·W2 + b2)
```

In [ ]:
def adelante(red, X):
    Z1 = ...
    A1 = ...
    P = ...
    return Z1, A1, P

_, _, P = adelante(red, X[:5])
print(P.round(2))           # sin entrenar: todo ~0.1, la red no sabe nada

In [ ]:
ok(P.shape == (5, 10) and np.allclose(P.sum(axis=1), 1), "adelante() da 10 probabilidades por imagen")

## 4. Hacia atrás: retropropagación
La idea, sin fórmulas: **el error viaja de la salida hacia la entrada, repartiéndose según los pesos.**

1. Error en la salida: `dZ2 = (P - Y) / n`. Igual que en la Clase 4.
2. Pendientes de la capa 2, igual que en la Clase 4: `dW2 = A1ᵀ · dZ2`, `db2 = suma de dZ2 por columnas`.
3. **¿Cuánta culpa tiene cada neurona oculta?** La que la conecta con la salida mediante su peso:
   `dA1 = dZ2 · W2ᵀ`.
4. Si la ReLU estaba apagada (`Z1 ≤ 0`), esa neurona no influyó → culpa 0: `dZ1 = dA1 * (Z1 > 0)`.
5. Pendientes de la capa 1, **con la misma receta**: `dW1 = Xᵀ · dZ1`, `db1 = suma de dZ1 por columnas`.

✏️ **Ejercicio 3.** Tradúcelo a código (cada paso es una línea).

In [ ]:
def gradientes(red, X, Y):
    Z1, A1, P = adelante(red, X)
    dZ2 = (P - Y) / len(X)
    dW2 = ...
    db2 = ...
    dA1 = ...
    dZ1 = ...
    dW1 = ...
    db1 = ...
    return {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}

**¿Cómo sé si mi retropropagación está bien?** Con la definición de pendiente: muevo **un** peso un poquito y
miro cuánto cambia el error. Si coincide con lo que calculó tu función, está bien. (Comprobación ya hecha.)

In [ ]:
def error_promedio(red, X, Y):
    return -np.mean(np.sum(Y * np.log(adelante(red, X)[2] + 1e-9), axis=1))

g = gradientes(red, X[:50], Y[:50])
for nombre, (i, j) in [("W1", (300, 5)), ("W2", (7, 3))]:
    copia = {k: v.copy() for k, v in red.items()}
    copia[nombre][i, j] += 1e-5
    numerica = (error_promedio(copia, X[:50], Y[:50]) - error_promedio(red, X[:50], Y[:50])) / 1e-5
    print(f"{nombre}[{i},{j}]  tu función: {g[nombre][i, j]: .6f}   a mano: {numerica: .6f}")
    ok(abs(numerica - g[nombre][i, j]) < 1e-4, f"la pendiente de {nombre} es correcta")

## 5. Entrenar
El ciclo es **el mismo de la Clase 4**: lotes, pendientes, paso. Solo cambió de dónde salen las pendientes.

In [ ]:
def precision(red, X, y):
    return (adelante(red, X)[2].argmax(axis=1) == y).mean()


def entrenar(red, epocas=5, eta=0.1, lote=64, semilla=0):
    rng = np.random.default_rng(semilla)
    for epoca in range(epocas):
        orden = rng.permutation(len(X))
        for i in range(0, len(X), lote):
            idx = orden[i:i + lote]
            g = gradientes(red, X[idx], Y[idx])
            for k in red:
                red[k] -= eta * g[k]
        print(f"época {epoca + 1}: precisión en el examen {precision(red, Xt, y_test):.2%}")
    return red

red = entrenar(crear_red())

In [ ]:
ok(precision(red, Xt, y_test) > 0.96, "la red multicapa supera el 96 %")

## 6. ¡A dibujar con tu red de verdad! ✍️
Guarda los pesos y en la terminal (carpeta `proyecto/`): `python dibujar.py`. Compárala con la de la Clase 4.

In [ ]:
guardar_pesos(**red)

In [ ]:
if os.path.exists("mi_numero.png"):                      # opción para Colab (ver Clase 4)
    x = leer_dibujo("mi_numero.png")
    p = adelante(red, x[None])[2][0]
    mostrar(x, titulos=[f"la red dice: {p.argmax()} ({p.max():.0%})"])

## 7. Experimentos 🔁 (elige uno y cuéntale a la clase)
- `crear_red(ocultas=16)` vs `ocultas=512`: ¿más neuronas siempre es mejor? ¿Y el tiempo?
- `eta=1.0` o `eta=0.001`: ¿qué pasa?
- Cambia `relu` por la sigmoide de la Clase 3 (¡y su derivada en `gradientes`!). ¿Aprende más lento?

## Lo que nos llevamos
- Una **capa oculta** aprende rasgos; la salida decide con ellos → de 92 % a ~97 %.
- **Retropropagación** = repartir el error hacia atrás, capa por capa, con la misma receta «error × entrada».
- Comprobar las pendientes numéricamente es cómo los profesionales depuran redes.

**Clase 6:** el entrenamiento tarda. ¿Se puede bajar la montaña más rápido? Optimizadores.